# 37 - System II Lognormal-Prior SBC Diagnostic

Hypothesis: the broad flat System II prior may push training and SBC validation into physically awkward regions, degrading calibration. This notebook tests a truncated lognormal prior centered on nominal operation and asks whether calibration and seed stability improve.

Design choices:
- Work on System II, S-B, matched per-draw steady-state protocol.
- Center the prior at nominal operation: `[alpha, beta_r, eta_col, xi_reb, z_A0_eff] = [1, 1, 1, 1, 0.90]`.
- Use independent lognormal marginals, truncated to the existing simulator support.
- Build a 5,000-sample approximate lognormal training bank by importance-resampling the existing full matched uniform bank.
- Train several seeds and run matched SBC with `reject_outside_prior=True`.

This is a diagnostic, not a production posterior. If this improves calibration, the next step should be a true simulator-generated lognormal bank rather than relying only on resampled uniform-bank points.

In [ ]:
import json, pathlib, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from cstr_sbi.recycle.physics import PARAM_NAMES
from cstr_sbi.recycle.priors import PRIOR_LOW_5D, PRIOR_HIGH_5D
from sbi_pipeline import _structure_config, _simulate_summary

DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
SBI_LOGS = ROOT / 'sbi-logs'
FIGURES = ROOT / 'figures'
for path in [RESULTS, SBI_LOGS, FIGURES]:
    path.mkdir(exist_ok=True)

FULL_BANK_PATH = DATA / 'wu2003_sbi_train_sb_n03_matched_full.npz'
assert FULL_BANK_PATH.exists(), FULL_BANK_PATH

STRUCTURE = 'S-B'
NOISE_PCT = 0.003
N_TRAIN = 5000
TORCH_SEEDS = [0, 1, 2, 3]
ARCH = 'zuko_nsf'
HIDDEN = 60
TRANSFORMS = 3
MAX_EPOCHS = 120
STOP_AFTER_EPOCHS = 12
N_SBC = 120
N_POST = 120
TRAIN_RESAMPLE_SEED = 20260827
SBC_SEED_BASE = 370000

NOMINAL_THETA = np.array([1.0, 1.0, 1.0, 1.0, 0.90], dtype=np.float32)
LOG_SIGMA = np.array([0.16, 0.16, 0.10, 0.16, 0.055], dtype=np.float32)

SUMMARY_PATH = RESULTS / '37_sysII_lognormal_prior_sbc.json'
CSV_PATH = RESULTS / '37_sysII_lognormal_prior_sbc.csv'
BANK_PATH = DATA / f'wu2003_sbi_train_sb_lognormal_resampled_n{N_TRAIN}.npz'

print(f'ROOT={ROOT}')
print(f'N_TRAIN={N_TRAIN}, seeds={TORCH_SEEDS}, N_SBC={N_SBC}, N_POST={N_POST}')
print(f'prior low={PRIOR_LOW_5D}')
print(f'prior high={PRIOR_HIGH_5D}')
print(f'nominal={NOMINAL_THETA}')
print(f'log sigma={LOG_SIGMA}')

## 1. Truncated lognormal prior

The prior is represented directly in physical parameter space. The lognormal median equals the nominal operating value, and values outside the current physical support are rejected.

In [ ]:
class TruncatedLogNormalPrior(torch.distributions.Distribution):
    arg_constraints = {}
    has_rsample = False

    def __init__(self, median, sigma, low, high, validate_args=False):
        self.median = torch.as_tensor(median, dtype=torch.float32)
        self.sigma = torch.as_tensor(sigma, dtype=torch.float32)
        self.low = torch.as_tensor(low, dtype=torch.float32)
        self.high = torch.as_tensor(high, dtype=torch.float32)
        self.mu = torch.log(self.median)
        self.base = torch.distributions.Normal(self.mu, self.sigma)
        lo_z = (torch.log(self.low) - self.mu) / self.sigma
        hi_z = (torch.log(self.high) - self.mu) / self.sigma
        standard = torch.distributions.Normal(torch.zeros_like(self.mu), torch.ones_like(self.mu))
        self.log_norm = torch.log(standard.cdf(hi_z) - standard.cdf(lo_z)).sum()
        super().__init__(batch_shape=torch.Size(), event_shape=torch.Size([len(self.median)]), validate_args=validate_args)

    @property
    def mean(self):
        draws = self.sample((20000,))
        return draws.mean(dim=0)

    @property
    def variance(self):
        draws = self.sample((20000,))
        return draws.var(dim=0)

    def sample(self, sample_shape=torch.Size()):
        if isinstance(sample_shape, int):
            sample_shape = torch.Size([sample_shape])
        sample_shape = torch.Size(sample_shape)
        n = int(np.prod(sample_shape)) if len(sample_shape) else 1
        accepted = []
        batch = max(1024, 2 * n)
        while sum(a.shape[0] for a in accepted) < n:
            z = self.base.sample((batch,))
            theta = torch.exp(z)
            mask = ((theta >= self.low) & (theta <= self.high)).all(dim=1)
            if mask.any():
                accepted.append(theta[mask])
        out = torch.cat(accepted, dim=0)[:n]
        return out.reshape(sample_shape + self.event_shape)

    def log_prob(self, value):
        value = torch.as_tensor(value, dtype=torch.float32)
        inside = ((value >= self.low) & (value <= self.high)).all(dim=-1)
        safe_value = torch.clamp(value, min=torch.finfo(torch.float32).tiny)
        logpdf = self.base.log_prob(torch.log(safe_value)).sum(dim=-1) - torch.log(safe_value).sum(dim=-1) - self.log_norm
        return torch.where(inside, logpdf, torch.full_like(logpdf, -torch.inf))

prior_logn = TruncatedLogNormalPrior(NOMINAL_THETA, LOG_SIGMA, PRIOR_LOW_5D, PRIOR_HIGH_5D)
samples_prior = prior_logn.sample((20000,)).numpy()
prior_df = pd.DataFrame(samples_prior, columns=PARAM_NAMES)
display(prior_df.describe(percentiles=[0.05, 0.5, 0.95]).T)

fig, axes = plt.subplots(1, len(PARAM_NAMES), figsize=(12, 2.2))
for ax, name, lo, hi, nominal in zip(axes, PARAM_NAMES, PRIOR_LOW_5D, PRIOR_HIGH_5D, NOMINAL_THETA):
    ax.hist(prior_df[name], bins=40, color='tab:blue', alpha=0.75)
    ax.axvline(nominal, color='k', lw=1.2, ls='--')
    ax.axvline(lo, color='0.4', lw=0.8, ls=':')
    ax.axvline(hi, color='0.4', lw=0.8, ls=':')
    ax.set_title(name, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / '37_sysII_lognormal_prior_marginals.png', dpi=180, bbox_inches='tight')
plt.show()

## 2. Importance-resample the existing matched bank

The completed full matched bank was drawn from the flat box prior. Because the flat density is constant inside the box, resampling rows with probability proportional to the truncated-lognormal density gives an approximate lognormal-prior bank without rerunning the simulator. The effective sample size is reported so we can see whether the resampled bank is too concentrated.

In [ ]:
full_bank = np.load(FULL_BANK_PATH)
theta_full = np.asarray(full_bank['thetas'], dtype=np.float32)
summary_full = np.asarray(full_bank['summaries'], dtype=np.float32)
logw = prior_logn.log_prob(torch.tensor(theta_full, dtype=torch.float32)).numpy()
logw = logw - np.max(logw)
w = np.exp(logw)
w = w / w.sum()
ess = 1.0 / np.sum(w ** 2)
print(f'Full bank={theta_full.shape}, lognormal importance ESS={ess:.0f} / {len(w)}')

rng = np.random.default_rng(TRAIN_RESAMPLE_SEED)
idx = rng.choice(len(theta_full), size=N_TRAIN, replace=True, p=w)
theta_train = theta_full[idx]
summary_train = summary_full[idx]
np.savez(BANK_PATH, thetas=theta_train, summaries=summary_train, source_bank=str(FULL_BANK_PATH), resample_indices=idx, weights=w)
print(f'wrote {BANK_PATH.name}: theta={theta_train.shape}, summaries={summary_train.shape}, unique rows={len(np.unique(idx))}')

compare = pd.DataFrame({
    'parameter': PARAM_NAMES,
    'flat_bank_mean': theta_full.mean(axis=0),
    'logn_resampled_mean': theta_train.mean(axis=0),
    'flat_bank_q05': np.percentile(theta_full, 5, axis=0),
    'logn_resampled_q05': np.percentile(theta_train, 5, axis=0),
    'flat_bank_q95': np.percentile(theta_full, 95, axis=0),
    'logn_resampled_q95': np.percentile(theta_train, 95, axis=0),
})
display(compare)
compare.to_csv(RESULTS / '37_sysII_lognormal_resampled_bank_summary.csv', index=False)

## 3. Train posteriors under the lognormal prior

In [ ]:
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn

def train_lognormal_posterior(torch_seed):
    torch.manual_seed(torch_seed)
    thetas_t = torch.tensor(theta_train, dtype=torch.float32)
    summaries_t = torch.tensor(summary_train, dtype=torch.float32)
    density_estimator = posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS)
    inference = SNPE(prior=prior_logn, density_estimator=density_estimator)
    inference.append_simulations(thetas_t, summaries_t)
    t0 = time.time()
    trained = inference.train(
        max_num_epochs=MAX_EPOCHS,
        validation_fraction=0.1,
        stop_after_epochs=STOP_AFTER_EPOCHS,
        show_train_summary=True,
    )
    posterior = inference.build_posterior(trained)
    return posterior, time.time() - t0

## 4. Matched SBC with strict prior rejection

Validation draws come from the same truncated lognormal prior. The simulator still uses the matched per-draw steady-state warm start. Posterior sampling uses `reject_outside_prior=True`.

In [ ]:
SBC_SET_PATH = RESULTS / f'37_sysII_lognormal_sbc_set_n{N_SBC}.npz'
ctrl_sbc, y0_sbc = _structure_config(STRUCTURE)

if SBC_SET_PATH.exists():
    sbc_set = np.load(SBC_SET_PATH)
    theta_sbc = np.asarray(sbc_set['theta'], dtype=np.float32)
    summary_sbc = np.asarray(sbc_set['summaries'], dtype=np.float32)
    print(f'loaded cached lognormal SBC set: theta={theta_sbc.shape}, summaries={summary_sbc.shape}')
else:
    rng = np.random.default_rng(SBC_SEED_BASE)
    theta_rows, summary_rows = [], []
    t0 = time.time()
    while len(theta_rows) < N_SBC:
        th = prior_logn.sample((1,)).numpy()[0]
        s = _simulate_summary(th, ctrl_sbc, y0_sbc, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
        if s is None:
            continue
        theta_rows.append(th)
        summary_rows.append(s)
        if len(theta_rows) % 20 == 0:
            print(f'generated {len(theta_rows)}/{N_SBC} lognormal matched SBC summaries ({time.time()-t0:.0f}s)')
    theta_sbc = np.asarray(theta_rows, dtype=np.float32)
    summary_sbc = np.asarray(summary_rows, dtype=np.float32)
    np.savez(SBC_SET_PATH, theta=theta_sbc, summaries=summary_sbc)
    print(f'saved lognormal SBC set: {SBC_SET_PATH}, wall_s={time.time()-t0:.0f}')

def run_lognormal_sbc(posterior, n_post=N_POST, seed=SBC_SEED_BASE):
    torch.manual_seed(seed)
    ranks = {name: [] for name in PARAM_NAMES}
    sample_seconds = []
    t0 = time.time()
    for i, (th, s) in enumerate(zip(theta_sbc, summary_sbc)):
        t_sample = time.time()
        with torch.no_grad():
            samp = posterior.sample(
                (n_post,),
                x=torch.tensor(s, dtype=torch.float32),
                show_progress_bars=False,
                reject_outside_prior=True,
            ).detach().cpu().numpy()
        sample_seconds.append(time.time() - t_sample)
        for k, name in enumerate(PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
        if (i + 1) % 20 == 0:
            print(f'[logn-sbc] {i+1}/{len(theta_sbc)} ({time.time()-t0:.0f}s, last_sample_s={sample_seconds[-1]:.3f})')

    results = {
        'n_sbc': int(len(theta_sbc)),
        'n_post': n_post,
        'n_valid': int(len(theta_sbc)),
        'reject_outside_prior': True,
        'shared_validation_set': str(SBC_SET_PATH),
        'sample_seconds_mean': float(np.mean(sample_seconds)) if sample_seconds else None,
        'sample_seconds_p95': float(np.percentile(sample_seconds, 95)) if sample_seconds else None,
        'sample_seconds_max': float(np.max(sample_seconds)) if sample_seconds else None,
    }
    for name in PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf) if len(r) else None
        results[name] = {
            'ranks': r.astype(int).tolist(),
            'ks_pvalue': float(ks.pvalue) if ks is not None else None,
            'mean_rank_frac': float(r.mean() / n_post) if len(r) else None,
        }
    results['min_ks_pvalue'] = min(results[name]['ks_pvalue'] for name in PARAM_NAMES)
    print(f'[logn-sbc] completed in {time.time()-t0:.0f}s; min KS p={results["min_ks_pvalue"]:.3e}')
    return results

def flatten_result(row):
    rec = {
        'seed': row['seed'],
        'n_train': row['n_train'],
        'train_wall_s': row['train_wall_s'],
        'sbc_wall_s': row['sbc_wall_s'],
        'min_ks_pvalue': row['sbc']['min_ks_pvalue'],
        'sample_seconds_mean': row['sbc']['sample_seconds_mean'],
        'sample_seconds_p95': row['sbc']['sample_seconds_p95'],
        'sample_seconds_max': row['sbc']['sample_seconds_max'],
    }
    for name in PARAM_NAMES:
        rec[f'{name}_ks_pvalue'] = row['sbc'][name]['ks_pvalue']
        rec[f'{name}_mean_rank_frac'] = row['sbc'][name]['mean_rank_frac']
    return rec

## 5. Seed-stability experiment

In [ ]:
all_results = []
t_all = time.time()
for seed in TORCH_SEEDS:
    variant = f'lognormal_prior_n{N_TRAIN}_seed{seed}'
    posterior_path = SBI_LOGS / f'wu2003_posterior_{variant}.pkl'
    print(f'\n=== {variant} ===')

    if posterior_path.exists():
        with open(posterior_path, 'rb') as f:
            payload = pickle.load(f)
        posterior = payload['posterior']
        train_wall_s = float(payload.get('train_wall_s', 0.0))
        print(f'loaded existing posterior: {posterior_path}')
    else:
        posterior, train_wall_s = train_lognormal_posterior(seed)
        with open(posterior_path, 'wb') as f:
            pickle.dump({
                'posterior': posterior,
                'variant': variant,
                'n_train': N_TRAIN,
                'torch_seed': seed,
                'arch': ARCH,
                'hidden_features': HIDDEN,
                'num_transforms': TRANSFORMS,
                'prior': 'truncated_lognormal_centered_nominal',
                'nominal_theta': NOMINAL_THETA.tolist(),
                'log_sigma': LOG_SIGMA.tolist(),
                'prior_low': PRIOR_LOW_5D.tolist(),
                'prior_high': PRIOR_HIGH_5D.tolist(),
                'bank_path': str(BANK_PATH),
                'train_wall_s': train_wall_s,
            }, f)
        print(f'saved posterior: {posterior_path}')

    t_sbc = time.time()
    sbc = run_lognormal_sbc(posterior, n_post=N_POST, seed=SBC_SEED_BASE + seed)
    sbc_wall_s = time.time() - t_sbc

    row = {
        'seed': seed,
        'variant': variant,
        'n_train': N_TRAIN,
        'train_wall_s': train_wall_s,
        'sbc_wall_s': sbc_wall_s,
        'posterior_path': str(posterior_path),
        'sbc': sbc,
    }
    all_results.append(row)
    summary = {
        'status': 'in_progress',
        'structure': STRUCTURE,
        'noise_pct': NOISE_PCT,
        'n_train': N_TRAIN,
        'completed_seeds': [r['seed'] for r in all_results],
        'torch_seeds': TORCH_SEEDS,
        'arch': ARCH,
        'hidden_features': HIDDEN,
        'num_transforms': TRANSFORMS,
        'max_epochs': MAX_EPOCHS,
        'stop_after_epochs': STOP_AFTER_EPOCHS,
        'n_sbc': int(len(theta_sbc)),
        'n_post': N_POST,
        'reject_outside_prior': True,
        'shared_validation_set': str(SBC_SET_PATH),
        'prior': {
            'type': 'truncated_lognormal_centered_nominal',
            'nominal_theta': NOMINAL_THETA.tolist(),
            'log_sigma': LOG_SIGMA.tolist(),
            'low': PRIOR_LOW_5D.tolist(),
            'high': PRIOR_HIGH_5D.tolist(),
        },
        'training_bank': {
            'path': str(BANK_PATH),
            'source_bank': str(FULL_BANK_PATH),
            'importance_ess': float(ess),
            'unique_rows': int(len(np.unique(idx))),
        },
        'results': all_results,
        'elapsed_wall_s': time.time() - t_all,
    }
    with open(SUMMARY_PATH, 'w') as f:
        json.dump(summary, f, indent=2)
    pd.DataFrame([flatten_result(r) for r in all_results]).to_csv(CSV_PATH, index=False)

final_summary = {**summary, 'status': 'complete'}
with open(SUMMARY_PATH, 'w') as f:
    json.dump(final_summary, f, indent=2)
result_df = pd.DataFrame([flatten_result(r) for r in all_results]).sort_values('min_ks_pvalue', ascending=False)
result_df.to_csv(CSV_PATH, index=False)
display(result_df)
print(f'total wall time: {(time.time() - t_all)/3600:.2f} h')

## 6. Plot seed stability

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5))
axes[0].bar(result_df['seed'].astype(str), result_df['min_ks_pvalue'], color='tab:green', alpha=0.78)
axes[0].axhline(0.05, color='k', ls='--', lw=1.0, label='pass threshold')
axes[0].set_yscale('log')
axes[0].set_xlabel('torch seed')
axes[0].set_ylabel('minimum KS p-value')
axes[0].set_title('Lognormal-prior SBC by seed')
axes[0].legend(fontsize=8)

rank_cols = [f'{name}_mean_rank_frac' for name in PARAM_NAMES]
rank_matrix = result_df.set_index('seed')[rank_cols].rename(columns={f'{name}_mean_rank_frac': name for name in PARAM_NAMES})
im = axes[1].imshow(rank_matrix.values, vmin=0.35, vmax=0.65, cmap='coolwarm', aspect='auto')
axes[1].set_xticks(range(len(PARAM_NAMES)))
axes[1].set_xticklabels(PARAM_NAMES, rotation=35, ha='right')
axes[1].set_yticks(range(len(rank_matrix.index)))
axes[1].set_yticklabels(rank_matrix.index)
axes[1].set_title('Mean rank fraction (0.5 ideal)')
fig.colorbar(im, ax=axes[1], shrink=0.8)
fig.tight_layout()
fig.savefig(FIGURES / '37_sysII_lognormal_prior_sbc_seed_stability.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Decision rule

Interpretation rule:

- If several seeds pass min-KS > 0.05 and mean rank fractions cluster near 0.5, the flat prior is likely a major contributor to the previous SBC failure. The next step should be a true lognormal-prior simulator bank, not just resampling from the flat bank.
- If only one seed passes, or if rank skew remains systematic, the prior helped but did not solve the issue; combine the lognormal prior with the summary-preprocessing fixes flagged in notebook 35.
- If strict rejection slows dramatically, support mismatch remains an issue and a bounded/support-aware parameterization should be tested before another production run.

## 8. True simulator-generated lognormal bank

The resampled-bank test above is useful but imperfect: it reweights a flat-prior simulator bank rather than drawing a new simulator bank from the lognormal prior itself. This section generates a true matched-protocol lognormal bank, trains multiple seeds, and evaluates them on the same shared lognormal SBC validation set.

In [ ]:
N_TRUE_TRAIN = 1000
TRUE_TRAIN_SEED = 20260828
TRUE_BANK_PATH = DATA / f'wu2003_sbi_train_sb_lognormal_true_n{N_TRUE_TRAIN}.npz'
TRUE_SUMMARY_PATH = RESULTS / f'37_sysII_lognormal_true_bank_n{N_TRUE_TRAIN}_sbc.json'
TRUE_CSV_PATH = RESULTS / f'37_sysII_lognormal_true_bank_n{N_TRUE_TRAIN}_sbc.csv'

print(f'True simulator-generated lognormal bank target: N_TRUE_TRAIN={N_TRUE_TRAIN}')
print(f'TRUE_BANK_PATH={TRUE_BANK_PATH}')
print(f'TRUE_SUMMARY_PATH={TRUE_SUMMARY_PATH}')

In [ ]:
def _load_partial_true_lognormal_bank(out_path, n_train):
    partial_candidates = [out_path.with_suffix('.partial.npz')]
    partial_candidates.extend(sorted(DATA.glob('wu2003_sbi_train_sb_lognormal_true_n*.partial.npz')))
    for partial in partial_candidates:
        if not partial.exists():
            continue
        bank = np.load(partial)
        theta_part = np.asarray(bank['thetas'], dtype=np.float32)
        summary_part = np.asarray(bank['summaries'], dtype=np.float32)
        if theta_part.ndim == 2 and summary_part.ndim == 2 and theta_part.shape[0] == summary_part.shape[0]:
            n_keep = min(theta_part.shape[0], n_train)
            attempts = int(np.asarray(bank['attempts']).item()) if 'attempts' in bank.files else n_keep
            print(f'resuming from partial bank {partial.name}: {n_keep}/{n_train} rows')
            return theta_part[:n_keep].tolist(), summary_part[:n_keep].tolist(), attempts
    return [], [], 0

def generate_true_lognormal_bank(out_path=TRUE_BANK_PATH, n_train=N_TRUE_TRAIN, seed=TRUE_TRAIN_SEED):
    if out_path.exists():
        bank = np.load(out_path)
        print(f'loading existing true lognormal bank: {out_path}')
        return np.asarray(bank['thetas'], dtype=np.float32), np.asarray(bank['summaries'], dtype=np.float32)

    ctrl, y0 = _structure_config(STRUCTURE)
    rng = np.random.default_rng(seed)
    theta_rows, summary_rows, attempts = _load_partial_true_lognormal_bank(out_path, n_train)
    t0 = time.time()
    while len(theta_rows) < n_train:
        attempts += 1
        th = prior_logn.sample((1,)).numpy()[0]
        s = _simulate_summary(th, ctrl, y0, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
        if s is None or not np.isfinite(s).all():
            continue
        theta_rows.append(th)
        summary_rows.append(s)
        if len(theta_rows) % 100 == 0:
            elapsed = time.time() - t0
            print(f'[true-logn-bank] {len(theta_rows)}/{n_train} valid, attempts={attempts}, elapsed={elapsed:.0f}s')
            np.savez(
                out_path.with_suffix('.partial.npz'),
                thetas=np.asarray(theta_rows, dtype=np.float32),
                summaries=np.asarray(summary_rows, dtype=np.float32),
                attempts=attempts,
                seed=seed,
                n_target=n_train,
            )

    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    np.savez(
        out_path,
        thetas=theta_arr,
        summaries=summary_arr,
        attempts=attempts,
        seed=seed,
        n_target=n_train,
        prior='truncated_lognormal_centered_nominal',
        nominal_theta=NOMINAL_THETA,
        log_sigma=LOG_SIGMA,
        prior_low=PRIOR_LOW_5D,
        prior_high=PRIOR_HIGH_5D,
    )
    print(f'saved true lognormal bank: {out_path}, theta={theta_arr.shape}, summaries={summary_arr.shape}, wall_s={time.time()-t0:.0f}')
    return theta_arr, summary_arr

true_theta_train, true_summary_train = generate_true_lognormal_bank()
true_bank_df = pd.DataFrame(true_theta_train, columns=PARAM_NAMES)
display(true_bank_df.describe(percentiles=[0.05, 0.5, 0.95]).T)
true_bank_df.describe(percentiles=[0.05, 0.5, 0.95]).T.to_csv(RESULTS / f'37_sysII_lognormal_true_bank_n{N_TRUE_TRAIN}_theta_summary.csv')

In [ ]:
def train_true_lognormal_posterior(torch_seed):
    torch.manual_seed(torch_seed)
    thetas_t = torch.tensor(true_theta_train, dtype=torch.float32)
    summaries_t = torch.tensor(true_summary_train, dtype=torch.float32)
    density_estimator = posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS)
    inference = SNPE(prior=prior_logn, density_estimator=density_estimator)
    inference.append_simulations(thetas_t, summaries_t)
    t0 = time.time()
    trained = inference.train(
        max_num_epochs=MAX_EPOCHS,
        validation_fraction=0.1,
        stop_after_epochs=STOP_AFTER_EPOCHS,
        show_train_summary=True,
    )
    posterior = inference.build_posterior(trained)
    return posterior, time.time() - t0

true_results = []
t_all = time.time()
for seed in TORCH_SEEDS:
    variant = f'lognormal_true_bank_n{N_TRUE_TRAIN}_seed{seed}'
    posterior_path = SBI_LOGS / f'wu2003_posterior_{variant}.pkl'
    print(f'\n=== {variant} ===')

    if posterior_path.exists():
        with open(posterior_path, 'rb') as f:
            payload = pickle.load(f)
        posterior = payload['posterior']
        train_wall_s = float(payload.get('train_wall_s', 0.0))
        print(f'loaded existing posterior: {posterior_path}')
    else:
        posterior, train_wall_s = train_true_lognormal_posterior(seed)
        with open(posterior_path, 'wb') as f:
            pickle.dump({
                'posterior': posterior,
                'variant': variant,
                'n_train': N_TRUE_TRAIN,
                'torch_seed': seed,
                'arch': ARCH,
                'hidden_features': HIDDEN,
                'num_transforms': TRANSFORMS,
                'prior': 'truncated_lognormal_centered_nominal',
                'bank_path': str(TRUE_BANK_PATH),
                'train_wall_s': train_wall_s,
            }, f)
        print(f'saved posterior: {posterior_path}')

    t_sbc = time.time()
    sbc = run_lognormal_sbc(posterior, n_post=N_POST, seed=SBC_SEED_BASE + 1000 + seed)
    sbc_wall_s = time.time() - t_sbc

    row = {
        'seed': seed,
        'variant': variant,
        'n_train': N_TRUE_TRAIN,
        'train_wall_s': train_wall_s,
        'sbc_wall_s': sbc_wall_s,
        'posterior_path': str(posterior_path),
        'sbc': sbc,
    }
    true_results.append(row)
    true_summary = {
        'status': 'in_progress',
        'structure': STRUCTURE,
        'noise_pct': NOISE_PCT,
        'n_train': N_TRUE_TRAIN,
        'completed_seeds': [r['seed'] for r in true_results],
        'torch_seeds': TORCH_SEEDS,
        'arch': ARCH,
        'hidden_features': HIDDEN,
        'num_transforms': TRANSFORMS,
        'max_epochs': MAX_EPOCHS,
        'stop_after_epochs': STOP_AFTER_EPOCHS,
        'n_sbc': int(len(theta_sbc)),
        'n_post': N_POST,
        'reject_outside_prior': True,
        'shared_validation_set': str(SBC_SET_PATH),
        'prior': {
            'type': 'truncated_lognormal_centered_nominal',
            'nominal_theta': NOMINAL_THETA.tolist(),
            'log_sigma': LOG_SIGMA.tolist(),
            'low': PRIOR_LOW_5D.tolist(),
            'high': PRIOR_HIGH_5D.tolist(),
        },
        'training_bank': {
            'path': str(TRUE_BANK_PATH),
            'type': 'true_simulator_generated_lognormal',
        },
        'results': true_results,
        'elapsed_wall_s': time.time() - t_all,
    }
    with open(TRUE_SUMMARY_PATH, 'w') as f:
        json.dump(true_summary, f, indent=2)
    pd.DataFrame([flatten_result(r) for r in true_results]).to_csv(TRUE_CSV_PATH, index=False)
    print(f'wrote incremental true-bank summary: {TRUE_SUMMARY_PATH}')

true_summary = {**true_summary, 'status': 'complete'}
with open(TRUE_SUMMARY_PATH, 'w') as f:
    json.dump(true_summary, f, indent=2)
true_result_df = pd.DataFrame([flatten_result(r) for r in true_results]).sort_values('min_ks_pvalue', ascending=False)
true_result_df.to_csv(TRUE_CSV_PATH, index=False)
display(true_result_df)
print(f'true-bank lognormal experiment wall time: {(time.time() - t_all)/3600:.2f} h')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5))
axes[0].bar(true_result_df['seed'].astype(str), true_result_df['min_ks_pvalue'], color='tab:purple', alpha=0.78)
axes[0].axhline(0.05, color='k', ls='--', lw=1.0, label='pass threshold')
axes[0].set_yscale('log')
axes[0].set_xlabel('torch seed')
axes[0].set_ylabel('minimum KS p-value')
axes[0].set_title('True lognormal-bank SBC by seed')
axes[0].legend(fontsize=8)

rank_cols = [f'{name}_mean_rank_frac' for name in PARAM_NAMES]
rank_matrix = true_result_df.set_index('seed')[rank_cols].rename(columns={f'{name}_mean_rank_frac': name for name in PARAM_NAMES})
im = axes[1].imshow(rank_matrix.values, vmin=0.35, vmax=0.65, cmap='coolwarm', aspect='auto')
axes[1].set_xticks(range(len(PARAM_NAMES)))
axes[1].set_xticklabels(PARAM_NAMES, rotation=35, ha='right')
axes[1].set_yticks(range(len(rank_matrix.index)))
axes[1].set_yticklabels(rank_matrix.index)
axes[1].set_title('Mean rank fraction (0.5 ideal)')
fig.colorbar(im, ax=axes[1], shrink=0.8)
fig.tight_layout()
fig.savefig(FIGURES / f'37_sysII_lognormal_true_bank_n{N_TRUE_TRAIN}_sbc_seed_stability.png', dpi=180, bbox_inches='tight')
plt.show()

print('True simulator-generated lognormal bank result:')
for _, row in true_result_df.iterrows():
    print(f"  seed {int(row['seed'])}: min KS p={row['min_ks_pvalue']:.3e}, train={row['train_wall_s']:.0f}s, SBC={row['sbc_wall_s']:.0f}s")

## 9. Next moves after the 5k true-lognormal run

The 5k true simulator-generated lognormal run is the right diagnostic upgrade over the resampled-bank test, but it still does not establish stable calibration. At this point, the most useful next moves are not simply bigger banks. The failure should be decomposed into three candidate causes:

1. **Summary preprocessing pathology.** Notebook 35 showed that `V_norm*` and `Q_j*` summaries dominate the heavy-tail/outlier diagnostics. A normalising flow trained with ordinary z-scoring can overreact to these dimensions. Test robust clipping/scaling, signed-log transforms, and dropping the worst outlier features.
2. **Support parameterisation.** `reject_outside_prior=True` is fast in the canary tests, so rejection itself is not the bottleneck. But bounded parameters may still be awkward for an unconstrained flow. A logit-space parameterisation is the principled follow-up if preprocessing alone helps but does not stabilize seeds.
3. **Warm-start/protocol geometry.** The matched warm start is scientifically correct for the ongoing-degradation protocol, but it may create a more nonlinear map from parameters to summaries than the fixed-nominal warm start. A warm-start ablation can tell whether calibration failure is intrinsic to System II summaries, or mostly introduced by per-draw steady-state initialization.

Run the following experiments in order. Stop early if a cheap diagnostic clearly fails; do not launch a new full-budget ensemble until a reduced test gives stable seed behaviour.

### Current 5k true-lognormal evidence

The user reran the true simulator-generated lognormal bank at 5,000 simulations. The result is still not calibrated: 0/4 seeds pass at `N_SBC=120`, `N_POST=120`, `reject_outside_prior=True`. The best seed reaches only min-KS `p = 2.21e-3`; the others are `1.10e-4`, `8.08e-6`, and `4.49e-10`. This means prior narrowing alone is not enough, even when the bank is generated from the lognormal prior rather than resampled from a flat-prior bank. The next code cells therefore test preprocessing and warm-start geometry as separate hypotheses.

## 10. Preprocessing experiment: robust summaries before changing the simulator

This is the highest-return next test. It keeps the prior, simulator protocol, architecture, and validation set fixed, and changes only the summary representation passed to SNPE. The variants are deliberately small:

- `identity`: current behaviour, included as a control.
- `robust_clip_scale`: clips each summary to `median ± 8 IQR`, then robust-scales by IQR.
- `signed_log_robust`: applies a signed log transform around the median, then robust-scales. This is intended for heavy-tailed features such as `V_norm*` and `Q_j*`.
- `drop_heavy_tail`: removes the worst heavy-tail features identified by notebook 35, then robust-scales the remainder.

Interpretation:

- If one variant gives multiple passing seeds and mean ranks near 0.5, the SBC issue is mostly representational/preprocessing.
- If no variant helps, the next move is bounded/logit parameterisation or reducing/remodelling the problematic System II summary channels.
- If dropping features helps but transforms do not, the feature definitions themselves are likely contaminating the posterior.

In [ ]:
from cstr_sbi.recycle.summaries import summary_names

PREPROCESS_TRAIN_PATH = DATA / 'wu2003_sbi_train_sb_lognormal_true_n5000.npz'
PREPROCESS_SBC_SET_PATH = RESULTS / '37_sysII_lognormal_sbc_set_n120.npz'
assert PREPROCESS_TRAIN_PATH.exists(), PREPROCESS_TRAIN_PATH
assert PREPROCESS_SBC_SET_PATH.exists(), PREPROCESS_SBC_SET_PATH

train_bank_pp = np.load(PREPROCESS_TRAIN_PATH)
theta_train_pp = np.asarray(train_bank_pp['thetas'], dtype=np.float32)
summary_train_pp_raw = np.asarray(train_bank_pp['summaries'], dtype=np.float32)
sbc_bank_pp = np.load(PREPROCESS_SBC_SET_PATH)
theta_sbc_pp = np.asarray(sbc_bank_pp['theta'], dtype=np.float32)
summary_sbc_pp_raw = np.asarray(sbc_bank_pp['summaries'], dtype=np.float32)
summary_name_list = summary_names(STRUCTURE)
assert summary_train_pp_raw.shape[1] == len(summary_name_list)
assert summary_sbc_pp_raw.shape[1] == len(summary_name_list)

HEAVY_TAIL_NAMES = [
    'V_norm_mean', 'V_norm_q_mean', 'V_norm_max', 'V_norm_min', 'Vn_final',
    'Q_j_min', 'Q_j_q_mean', 'T_r_std', 'V_norm_std', 'F_R_norm_slope',
    'V_norm_slope', 'F_R_norm_std',
]
HEAVY_TAIL_IDXS = [i for i, name in enumerate(summary_name_list) if name in HEAVY_TAIL_NAMES]
print(f'Loaded preprocessing inputs: train={summary_train_pp_raw.shape}, sbc={summary_sbc_pp_raw.shape}')
print('Heavy-tail feature indices:', list(zip(HEAVY_TAIL_IDXS, [summary_name_list[i] for i in HEAVY_TAIL_IDXS])))

class SummaryPreprocessor:
    def __init__(self, variant, heavy_tail_idxs=None, clip_iqr=8.0, eps=1e-6):
        self.variant = variant
        self.heavy_tail_idxs = np.asarray(heavy_tail_idxs or [], dtype=int)
        self.clip_iqr = float(clip_iqr)
        self.eps = float(eps)
        self.keep_idxs = None

    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.median = np.median(x, axis=0).astype(np.float32)
        q1 = np.percentile(x, 25, axis=0).astype(np.float32)
        q3 = np.percentile(x, 75, axis=0).astype(np.float32)
        self.iqr = np.maximum(q3 - q1, self.eps).astype(np.float32)
        if self.variant == 'drop_heavy_tail':
            drop = set(self.heavy_tail_idxs.tolist())
            self.keep_idxs = np.asarray([i for i in range(x.shape[1]) if i not in drop], dtype=int)
        else:
            self.keep_idxs = np.arange(x.shape[1], dtype=int)
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        z = x.copy()
        if self.variant == 'identity':
            return z
        if self.variant == 'robust_clip_scale':
            lo = self.median - self.clip_iqr * self.iqr
            hi = self.median + self.clip_iqr * self.iqr
            z = np.clip(z, lo, hi)
            z = (z - self.median) / self.iqr
            return z[:, self.keep_idxs].astype(np.float32)
        if self.variant == 'signed_log_robust':
            centered = (z - self.median) / self.iqr
            z = np.sign(centered) * np.log1p(np.abs(centered))
            return z[:, self.keep_idxs].astype(np.float32)
        if self.variant == 'drop_heavy_tail':
            lo = self.median - self.clip_iqr * self.iqr
            hi = self.median + self.clip_iqr * self.iqr
            z = np.clip(z, lo, hi)
            z = (z - self.median) / self.iqr
            return z[:, self.keep_idxs].astype(np.float32)
        raise ValueError(f'Unknown preprocessing variant: {self.variant}')

    def metadata(self):
        return {
            'variant': self.variant,
            'clip_iqr': self.clip_iqr,
            'heavy_tail_names': [summary_name_list[i] for i in self.heavy_tail_idxs],
            'n_input_features': int(len(self.median)),
            'n_output_features': int(len(self.keep_idxs)),
        }

PREPROCESS_VARIANTS = ['identity', 'robust_clip_scale', 'signed_log_robust', 'drop_heavy_tail']
PREPROCESS_SEEDS = [0, 1, 2]
print('Preprocessing variants:', PREPROCESS_VARIANTS)
print('Seeds:', PREPROCESS_SEEDS)

In [ ]:
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn

PREPROCESS_SUMMARY_PATH = RESULTS / '37_sysII_lognormal_true_n5000_preprocess_sbc.json'
PREPROCESS_CSV_PATH = RESULTS / '37_sysII_lognormal_true_n5000_preprocess_sbc.csv'

# For a first pass, keep this small enough to run interactively. Increase after a variant looks promising.
PREPROCESS_N_SBC = min(120, len(theta_sbc_pp))
PREPROCESS_N_POST = 120


def _posterior_nn_for_preprocess(variant):
    if variant == 'identity':
        return posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS)
    try:
        return posterior_nn(
            model=ARCH,
            hidden_features=HIDDEN,
            num_transforms=TRANSFORMS,
            z_score_x='none',
        )
    except TypeError:
        print('posterior_nn does not accept z_score_x in this sbi version; falling back to default internal x-scaling')
        return posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS)


def train_preprocessed_lognormal_posterior(x_train, torch_seed, variant):
    torch.manual_seed(torch_seed)
    thetas_t = torch.tensor(theta_train_pp, dtype=torch.float32)
    summaries_t = torch.tensor(x_train, dtype=torch.float32)
    density_estimator = _posterior_nn_for_preprocess(variant)
    inference = SNPE(prior=prior_logn, density_estimator=density_estimator)
    inference.append_simulations(thetas_t, summaries_t)
    t0 = time.time()
    trained = inference.train(
        max_num_epochs=MAX_EPOCHS,
        validation_fraction=0.1,
        stop_after_epochs=STOP_AFTER_EPOCHS,
        show_train_summary=True,
    )
    posterior = inference.build_posterior(trained)
    return posterior, time.time() - t0


def run_preprocessed_sbc(posterior, x_sbc, seed, n_post=PREPROCESS_N_POST):
    torch.manual_seed(seed)
    ranks = {name: [] for name in PARAM_NAMES}
    sample_seconds = []
    t0 = time.time()
    for i, (th, x_obs) in enumerate(zip(theta_sbc_pp[:PREPROCESS_N_SBC], x_sbc[:PREPROCESS_N_SBC])):
        t_sample = time.time()
        with torch.no_grad():
            samp = posterior.sample(
                (n_post,),
                x=torch.tensor(x_obs, dtype=torch.float32),
                show_progress_bars=False,
                reject_outside_prior=True,
            ).detach().cpu().numpy()
        sample_seconds.append(time.time() - t_sample)
        for k, name in enumerate(PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
        if (i + 1) % 20 == 0:
            print(f'[preprocess-sbc] {i+1}/{PREPROCESS_N_SBC} ({time.time()-t0:.0f}s, last_sample_s={sample_seconds[-1]:.3f})')

    results = {
        'n_sbc': int(PREPROCESS_N_SBC),
        'n_post': int(n_post),
        'n_valid': int(PREPROCESS_N_SBC),
        'reject_outside_prior': True,
        'sample_seconds_mean': float(np.mean(sample_seconds)),
        'sample_seconds_p95': float(np.percentile(sample_seconds, 95)),
        'sample_seconds_max': float(np.max(sample_seconds)),
    }
    for name in PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
        results[name] = {
            'ranks': r.astype(int).tolist(),
            'ks_pvalue': float(ks.pvalue),
            'mean_rank_frac': float(r.mean() / n_post),
        }
    results['min_ks_pvalue'] = min(results[name]['ks_pvalue'] for name in PARAM_NAMES)
    return results


def flatten_preprocess_result(row):
    rec = {
        'preprocess': row['preprocess'],
        'seed': row['seed'],
        'n_train': row['n_train'],
        'n_features': row['n_features'],
        'train_wall_s': row['train_wall_s'],
        'sbc_wall_s': row['sbc_wall_s'],
        'min_ks_pvalue': row['sbc']['min_ks_pvalue'],
        'sample_seconds_mean': row['sbc']['sample_seconds_mean'],
        'sample_seconds_p95': row['sbc']['sample_seconds_p95'],
    }
    for name in PARAM_NAMES:
        rec[f'{name}_ks_pvalue'] = row['sbc'][name]['ks_pvalue']
        rec[f'{name}_mean_rank_frac'] = row['sbc'][name]['mean_rank_frac']
    return rec

print(f'Preprocessing SBC config: N_SBC={PREPROCESS_N_SBC}, N_POST={PREPROCESS_N_POST}')

In [ ]:
preprocess_results = []
t_all = time.time()
for variant in PREPROCESS_VARIANTS:
    pp = SummaryPreprocessor(variant, HEAVY_TAIL_IDXS).fit(summary_train_pp_raw)
    x_train_variant = pp.transform(summary_train_pp_raw)
    x_sbc_variant = pp.transform(summary_sbc_pp_raw)
    variant_bank_path = DATA / f'wu2003_sbi_train_sb_lognormal_true_n5000_{variant}.npz'
    np.savez(
        variant_bank_path,
        thetas=theta_train_pp,
        summaries=x_train_variant,
        preprocess=json.dumps(pp.metadata()),
        source_bank=str(PREPROCESS_TRAIN_PATH),
    )
    print(f'\n=== preprocessing variant: {variant} ({x_train_variant.shape[1]} features) ===')

    for seed in PREPROCESS_SEEDS:
        name = f'lognormal_true_n5000_{variant}_seed{seed}'
        posterior_path = SBI_LOGS / f'wu2003_posterior_{name}.pkl'
        if posterior_path.exists():
            with open(posterior_path, 'rb') as f:
                payload = pickle.load(f)
            posterior = payload['posterior']
            train_wall_s = float(payload.get('train_wall_s', 0.0))
            print(f'loaded existing posterior: {posterior_path.name}')
        else:
            posterior, train_wall_s = train_preprocessed_lognormal_posterior(x_train_variant, seed, variant)
            with open(posterior_path, 'wb') as f:
                pickle.dump({
                    'posterior': posterior,
                    'variant': name,
                    'preprocess': pp.metadata(),
                    'n_train': int(theta_train_pp.shape[0]),
                    'torch_seed': seed,
                    'prior': 'truncated_lognormal_centered_nominal',
                    'bank_path': str(variant_bank_path),
                    'train_wall_s': train_wall_s,
                }, f)
            print(f'saved posterior: {posterior_path.name}')

        t_sbc = time.time()
        sbc = run_preprocessed_sbc(posterior, x_sbc_variant, seed=410000 + 100 * PREPROCESS_VARIANTS.index(variant) + seed)
        sbc_wall_s = time.time() - t_sbc
        row = {
            'preprocess': variant,
            'seed': seed,
            'n_train': int(theta_train_pp.shape[0]),
            'n_features': int(x_train_variant.shape[1]),
            'train_wall_s': train_wall_s,
            'sbc_wall_s': sbc_wall_s,
            'posterior_path': str(posterior_path),
            'sbc': sbc,
        }
        preprocess_results.append(row)
        summary = {
            'status': 'in_progress',
            'experiment': 'lognormal_true_n5000_preprocessing_sweep',
            'structure': STRUCTURE,
            'noise_pct': NOISE_PCT,
            'preprocess_variants': PREPROCESS_VARIANTS,
            'completed': [{'preprocess': r['preprocess'], 'seed': r['seed']} for r in preprocess_results],
            'seeds': PREPROCESS_SEEDS,
            'n_train': int(theta_train_pp.shape[0]),
            'n_sbc': int(PREPROCESS_N_SBC),
            'n_post': int(PREPROCESS_N_POST),
            'reject_outside_prior': True,
            'training_bank': str(PREPROCESS_TRAIN_PATH),
            'shared_validation_set': str(PREPROCESS_SBC_SET_PATH),
            'heavy_tail_names': [summary_name_list[i] for i in HEAVY_TAIL_IDXS],
            'results': preprocess_results,
            'elapsed_wall_s': time.time() - t_all,
        }
        with open(PREPROCESS_SUMMARY_PATH, 'w') as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([flatten_preprocess_result(r) for r in preprocess_results]).to_csv(PREPROCESS_CSV_PATH, index=False)

summary = {**summary, 'status': 'complete'}
with open(PREPROCESS_SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
preprocess_df = pd.DataFrame([flatten_preprocess_result(r) for r in preprocess_results])
preprocess_df.to_csv(PREPROCESS_CSV_PATH, index=False)
display(preprocess_df.sort_values(['min_ks_pvalue'], ascending=False))
print(f'preprocessing sweep wall time: {(time.time() - t_all)/3600:.2f} h')

In [ ]:
if 'preprocess_df' in globals():
    summary_by_variant = (preprocess_df.groupby('preprocess')['min_ks_pvalue']
                          .agg(['count', 'max', 'median', 'min'])
                          .sort_values('max', ascending=False))
    display(summary_by_variant)

    fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
    for variant, g in preprocess_df.groupby('preprocess'):
        axes[0].scatter([variant] * len(g), g['min_ks_pvalue'], s=45, alpha=0.8)
    axes[0].axhline(0.05, color='k', ls='--', lw=1.0)
    axes[0].set_yscale('log')
    axes[0].set_ylabel('minimum KS p-value')
    axes[0].set_title('Preprocessing sweep: seed stability')
    axes[0].tick_params(axis='x', rotation=25)

    best = preprocess_df.sort_values('min_ks_pvalue', ascending=False).iloc[0]
    rank_values = [best[f'{name}_mean_rank_frac'] for name in PARAM_NAMES]
    axes[1].bar(PARAM_NAMES, rank_values, color='tab:blue', alpha=0.75)
    axes[1].axhline(0.5, color='k', ls='--', lw=1.0)
    axes[1].set_ylim(0.35, 0.65)
    axes[1].set_ylabel('mean rank fraction')
    axes[1].set_title(f"Best run: {best['preprocess']} seed {int(best['seed'])}")
    axes[1].tick_params(axis='x', rotation=30)

    fig.tight_layout()
    fig.savefig(FIGURES / '37_sysII_lognormal_true_n5000_preprocess_sbc.png', dpi=180, bbox_inches='tight')
    plt.show()

    print('Decision guide:')
    print('- Require more than one seed with min KS p > 0.05 before treating a preprocessing variant as stable.')
    print('- If only drop_heavy_tail improves, revise or remove those feature definitions before publication use.')
    print('- If robust_clip_scale or signed_log_robust improves, rerun that variant at larger SBC and then full budget.')
else:
    print('Run the preprocessing sweep cell above first.')

## 11. Warm-start ablation: matched versus fixed nominal

This ablation asks whether the matched per-draw steady-state warm start is making the inference map substantially harder. It is **not** a proposal to replace the manuscript protocol. Instead, it compares two internally consistent protocols:

- matched training + matched validation: the current ongoing-degradation protocol.
- fixed-nominal training + fixed-nominal validation: a simpler geometry diagnostic.

Interpretation:

- If fixed-nominal calibrates while matched does not, the hard part is the steady-state warm-start map. Then inspect which summaries are most sensitive to `_matched_y0(theta)` and consider a less extreme prior/domain or transformed steady-state features.
- If fixed-nominal also fails, the failure is more likely in the summary representation, flow parameterization, or weak/ambiguous information for `xi_reb` and `z_A0_eff`.
- Do not mix fixed training with matched validation for manuscript numbers; that would recreate the protocol mismatch problem.

In [ ]:
WARMSTART_N_TRAIN = 3000
WARMSTART_N_SBC = 120
WARMSTART_SEEDS = [0, 1, 2]
WARMSTART_BANK_PATH = DATA / f'wu2003_sbi_train_sb_lognormal_fixed_y0_n{WARMSTART_N_TRAIN}.npz'
WARMSTART_SBC_PATH = RESULTS / f'37_sysII_lognormal_fixed_y0_sbc_set_n{WARMSTART_N_SBC}.npz'
WARMSTART_SUMMARY_PATH = RESULTS / f'37_sysII_lognormal_fixed_y0_n{WARMSTART_N_TRAIN}_sbc.json'
WARMSTART_CSV_PATH = RESULTS / f'37_sysII_lognormal_fixed_y0_n{WARMSTART_N_TRAIN}_sbc.csv'


def generate_lognormal_bank_for_warm_start(out_path, n_rows, seed, scenario_specific_warm_start):
    if out_path.exists():
        bank = np.load(out_path)
        print(f'loading existing bank: {out_path}')
        return np.asarray(bank['thetas'], dtype=np.float32), np.asarray(bank['summaries'], dtype=np.float32)

    ctrl, y0 = _structure_config(STRUCTURE)
    rng = np.random.default_rng(seed)
    theta_rows, summary_rows = [], []
    t0 = time.time()
    attempts = 0
    while len(theta_rows) < n_rows:
        attempts += 1
        th = prior_logn.sample((1,)).numpy()[0]
        s = _simulate_summary(
            th,
            ctrl,
            y0,
            STRUCTURE,
            NOISE_PCT,
            rng,
            scenario_specific_warm_start=scenario_specific_warm_start,
        )
        if s is None or not np.isfinite(s).all():
            continue
        theta_rows.append(th)
        summary_rows.append(s)
        if len(theta_rows) % 250 == 0:
            print(f'[warm-start-bank] {len(theta_rows)}/{n_rows}, attempts={attempts}, elapsed={time.time()-t0:.0f}s')
            np.savez(
                out_path.with_suffix('.partial.npz'),
                thetas=np.asarray(theta_rows, dtype=np.float32),
                summaries=np.asarray(summary_rows, dtype=np.float32),
                attempts=attempts,
                seed=seed,
                scenario_specific_warm_start=scenario_specific_warm_start,
            )

    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    np.savez(
        out_path,
        thetas=theta_arr,
        summaries=summary_arr,
        attempts=attempts,
        seed=seed,
        scenario_specific_warm_start=scenario_specific_warm_start,
        prior='truncated_lognormal_centered_nominal',
    )
    print(f'saved bank: {out_path}, theta={theta_arr.shape}, summaries={summary_arr.shape}, wall_s={time.time()-t0:.0f}')
    return theta_arr, summary_arr

fixed_theta_train, fixed_summary_train = generate_lognormal_bank_for_warm_start(
    WARMSTART_BANK_PATH,
    WARMSTART_N_TRAIN,
    seed=20260829,
    scenario_specific_warm_start=False,
)
fixed_theta_sbc, fixed_summary_sbc = generate_lognormal_bank_for_warm_start(
    WARMSTART_SBC_PATH,
    WARMSTART_N_SBC,
    seed=20260830,
    scenario_specific_warm_start=False,
)
print('Fixed-y0 warm-start ablation data ready:', fixed_theta_train.shape, fixed_summary_train.shape, fixed_theta_sbc.shape, fixed_summary_sbc.shape)

In [ ]:
def train_fixed_y0_lognormal_posterior(torch_seed):
    torch.manual_seed(torch_seed)
    thetas_t = torch.tensor(fixed_theta_train, dtype=torch.float32)
    summaries_t = torch.tensor(fixed_summary_train, dtype=torch.float32)
    density_estimator = posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS)
    inference = SNPE(prior=prior_logn, density_estimator=density_estimator)
    inference.append_simulations(thetas_t, summaries_t)
    t0 = time.time()
    trained = inference.train(
        max_num_epochs=MAX_EPOCHS,
        validation_fraction=0.1,
        stop_after_epochs=STOP_AFTER_EPOCHS,
        show_train_summary=True,
    )
    posterior = inference.build_posterior(trained)
    return posterior, time.time() - t0


def run_fixed_y0_lognormal_sbc(posterior, n_post=N_POST, seed=20260831):
    torch.manual_seed(seed)
    ranks = {name: [] for name in PARAM_NAMES}
    sample_seconds = []
    for i, (th, x_obs) in enumerate(zip(fixed_theta_sbc, fixed_summary_sbc)):
        t_sample = time.time()
        with torch.no_grad():
            samp = posterior.sample(
                (n_post,),
                x=torch.tensor(x_obs, dtype=torch.float32),
                show_progress_bars=False,
                reject_outside_prior=True,
            ).detach().cpu().numpy()
        sample_seconds.append(time.time() - t_sample)
        for k, name in enumerate(PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
        if (i + 1) % 20 == 0:
            print(f'[fixed-y0-sbc] {i+1}/{len(fixed_theta_sbc)} last_sample_s={sample_seconds[-1]:.3f}')

    results = {
        'n_sbc': int(len(fixed_theta_sbc)),
        'n_post': int(n_post),
        'n_valid': int(len(fixed_theta_sbc)),
        'reject_outside_prior': True,
        'scenario_specific_warm_start': False,
        'sample_seconds_mean': float(np.mean(sample_seconds)),
        'sample_seconds_p95': float(np.percentile(sample_seconds, 95)),
        'sample_seconds_max': float(np.max(sample_seconds)),
    }
    for name in PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
        results[name] = {
            'ranks': r.astype(int).tolist(),
            'ks_pvalue': float(ks.pvalue),
            'mean_rank_frac': float(r.mean() / n_post),
        }
    results['min_ks_pvalue'] = min(results[name]['ks_pvalue'] for name in PARAM_NAMES)
    return results

fixed_results = []
t_all = time.time()
for seed in WARMSTART_SEEDS:
    variant = f'lognormal_fixed_y0_n{WARMSTART_N_TRAIN}_seed{seed}'
    posterior_path = SBI_LOGS / f'wu2003_posterior_{variant}.pkl'
    print(f'\n=== {variant} ===')
    if posterior_path.exists():
        with open(posterior_path, 'rb') as f:
            payload = pickle.load(f)
        posterior = payload['posterior']
        train_wall_s = float(payload.get('train_wall_s', 0.0))
        print(f'loaded existing posterior: {posterior_path.name}')
    else:
        posterior, train_wall_s = train_fixed_y0_lognormal_posterior(seed)
        with open(posterior_path, 'wb') as f:
            pickle.dump({
                'posterior': posterior,
                'variant': variant,
                'n_train': WARMSTART_N_TRAIN,
                'torch_seed': seed,
                'prior': 'truncated_lognormal_centered_nominal',
                'bank_path': str(WARMSTART_BANK_PATH),
                'scenario_specific_warm_start': False,
                'train_wall_s': train_wall_s,
            }, f)
        print(f'saved posterior: {posterior_path.name}')

    t_sbc = time.time()
    sbc = run_fixed_y0_lognormal_sbc(posterior, n_post=N_POST, seed=20260831 + seed)
    sbc_wall_s = time.time() - t_sbc
    row = {
        'seed': seed,
        'variant': variant,
        'n_train': WARMSTART_N_TRAIN,
        'train_wall_s': train_wall_s,
        'sbc_wall_s': sbc_wall_s,
        'posterior_path': str(posterior_path),
        'sbc': sbc,
    }
    fixed_results.append(row)
    fixed_summary = {
        'status': 'in_progress',
        'experiment': 'lognormal_fixed_nominal_warm_start_ablation',
        'structure': STRUCTURE,
        'noise_pct': NOISE_PCT,
        'n_train': WARMSTART_N_TRAIN,
        'n_sbc': int(len(fixed_theta_sbc)),
        'n_post': N_POST,
        'completed_seeds': [r['seed'] for r in fixed_results],
        'seeds': WARMSTART_SEEDS,
        'reject_outside_prior': True,
        'scenario_specific_warm_start': False,
        'training_bank': str(WARMSTART_BANK_PATH),
        'validation_bank': str(WARMSTART_SBC_PATH),
        'results': fixed_results,
        'elapsed_wall_s': time.time() - t_all,
    }
    with open(WARMSTART_SUMMARY_PATH, 'w') as f:
        json.dump(fixed_summary, f, indent=2)
    pd.DataFrame([flatten_result(r) for r in fixed_results]).to_csv(WARMSTART_CSV_PATH, index=False)

fixed_summary = {**fixed_summary, 'status': 'complete'}
with open(WARMSTART_SUMMARY_PATH, 'w') as f:
    json.dump(fixed_summary, f, indent=2)
fixed_df = pd.DataFrame([flatten_result(r) for r in fixed_results]).sort_values('min_ks_pvalue', ascending=False)
fixed_df.to_csv(WARMSTART_CSV_PATH, index=False)
display(fixed_df)
print(f'fixed-y0 warm-start ablation wall time: {(time.time() - t_all)/3600:.2f} h')

## 12. How to read the follow-up experiments

Use these criteria before investing in another full run:

1. **Seed stability beats one lucky seed.** A useful move should give at least 2/3 seeds passing min-KS > 0.05 at the reduced setting. One passing seed is only a hint.
2. **Look at the failing parameter, not just the minimum.** If only `xi_reb` or `z_A0_eff` remains bad, the issue may be specific to weak reboiler/feed-composition observability. If all parameters shift together, the problem is more likely global scaling or training instability.
3. **Watch the rank direction.** Mean rank fractions below 0.5 mean posterior samples are often above the truth; above 0.5 means they are often below the truth. Consistent direction across seeds is more diagnostic than the p-value alone.
4. **Preprocessing success path.** If `robust_clip_scale` or `signed_log_robust` improves seed stability, rerun that variant with larger `N_SBC` and then combine it with the best prior choice. If `drop_heavy_tail` is the only successful variant, revise the physical summary definitions rather than silently dropping them for the final paper.
5. **Warm-start success path.** If fixed-nominal warm start calibrates while matched warm start does not, the matched steady-state map is the hard part. Then compare summary distributions under matched and fixed warm starts, especially `V_norm*`, `Q_j*`, `xi_reb`, and `z_A0_eff` sensitivity.
6. **Failure path.** If both preprocessing and fixed-y0 ablations fail, the next serious move is bounded/logit parameter training and/or reducing System II to the identifiable parameter subset before rebuilding the final narrative.